# 🏦 Credit Risk Analyzer — Preprocessing Pipeline
## Home Credit Default Risk Dataset
- **Goal:** Clean the data, fix anomalies, engineer features, and prepare for ML modeling
- **Input:** application_train.csv (307,511 rows × 122 columns)
- **Output:** processed_data.csv — ready for XGBoost training

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Saving processed data
import os

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [2]:
# Load raw data
df = pd.read_csv('../data/raw/application_train.csv')
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Shape: (307511, 122)
Rows: 307,511
Columns: 122


In [3]:
# Step 1 — Fix DAYS_EMPLOYED anomaly
# 365243 = placeholder for unemployed/retired applicants (~1000 years)
# Replace with NaN — permanent fix in preprocessing pipeline

before = (df['DAYS_EMPLOYED'] == 365243).sum()
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
after = df['DAYS_EMPLOYED'].isnull().sum() - (df['DAYS_EMPLOYED'].isnull().sum() - before)

print(f"365243 values replaced: {before:,}")
print(f"DAYS_EMPLOYED null count now: {df['DAYS_EMPLOYED'].isnull().sum():,}")
print("✅ DAYS_EMPLOYED anomaly fixed!")

365243 values replaced: 55,374
DAYS_EMPLOYED null count now: 55,374
✅ DAYS_EMPLOYED anomaly fixed!


In [7]:
# Step 2 — Feature Engineering
# DAYS_BIRTH and DAYS_EMPLOYED are stored as negative days — convert to years

df['AGE_YEARS'] = (df['DAYS_BIRTH'] / -365).astype(int)
df['EMPLOYED_YEARS'] = (df['DAYS_EMPLOYED'] / -365).round(1)
# DAYS_EMPLOYED has NaN values (365243 replaced) — NaN will be preserved here

print("New features created:")
print(f"AGE_YEARS — min: {df['AGE_YEARS'].min()}, max: {df['AGE_YEARS'].max()}")
print(f"EMPLOYED_YEARS — min: {df['EMPLOYED_YEARS'].min():.1f}, max: {df['EMPLOYED_YEARS'].max():.1f}")
print(f"EMPLOYED_YEARS null count: {df['EMPLOYED_YEARS'].isnull().sum():,}")
print("✅ Feature engineering done!")

# Defragment the DataFrame for better performance
df = df.copy()

New features created:
AGE_YEARS — min: 20, max: 69
EMPLOYED_YEARS — min: 0.0, max: 49.1
EMPLOYED_YEARS null count: 55,374
✅ Feature engineering done!


In [8]:
# Step 3 — Create EMI ratio feature
# emi_ratio = monthly EMI (AMT_ANNUITY) / monthly income (AMT_INCOME_TOTAL)
# This directly maps to our rule engine logic

df['EMI_RATIO'] = (df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']).round(4)

print("EMI_RATIO feature created:")
print(f"Min: {df['EMI_RATIO'].min():.4f}")
print(f"Max: {df['EMI_RATIO'].max():.4f}")
print(f"Mean: {df['EMI_RATIO'].mean():.4f}")
print(f"Null count: {df['EMI_RATIO'].isnull().sum():,}")
print("✅ EMI_RATIO feature created!")

EMI_RATIO feature created:
Min: 0.0002
Max: 1.8760
Mean: 0.1809
Null count: 12
✅ EMI_RATIO feature created!


In [9]:
# Step 4 — Outlier capping using 99th percentile (Winsorization)
# Extreme outliers can distort model training — cap them at 99th percentile

cols_to_cap = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']

for col in cols_to_cap:
    cap_value = df[col].quantile(0.99)
    before_max = df[col].max()
    df[col] = df[col].clip(upper=cap_value)
    after_max = df[col].max()
    print(f"{col}:")
    print(f"  Before max: {before_max:,.0f} → After max: {after_max:,.0f}")

print("\n✅ Outlier capping done!")

AMT_INCOME_TOTAL:
  Before max: 117,000,000 → After max: 472,500
AMT_CREDIT:
  Before max: 4,050,000 → After max: 1,854,000
AMT_ANNUITY:
  Before max: 258,026 → After max: 70,006

✅ Outlier capping done!
